# Flow Matching (Rectified Flow) with Euler Sampling — Solution

**Difficulty**: 🟡 Medium

---

### Problem Statement

**Flow matching** (Lipman et al., 2023) and **rectified flow** (Liu et al., 2023) learn a time-dependent **velocity field** $v_\theta(x, t)$ whose ODE $\frac{dx}{dt} = v_\theta(x, t)$ carries Gaussian noise at $t = 0$ to data at $t = 1$. With the straight-line (optimal-transport) path between a noise sample and a data sample, training is plain regression — no noise schedule, no ELBO — and sampling is ODE integration. This is the objective behind **Stable Diffusion 3, Flux** and several recent video models, and "how does flow matching relate to diffusion?" is now a standard generative-modeling interview question.

**Convention used throughout** (state it in interviews — papers differ):

| | |
|---|---|
| noise / data | $x_0 \sim \mathcal N(0, I)$ at $t = 0$, $\;x_1 \sim p_{\text{data}}$ at $t = 1$ |
| path | $x_t = (1 - t)\,x_0 + t\,x_1$ |
| target velocity | $u = \dfrac{d x_t}{dt} = x_1 - x_0$ |
| loss | $\mathcal L(\theta) = \mathbb{E}_{t \sim U(0,1),\,x_0,\,x_1}\,\big\lVert v_\theta(x_t, t) - (x_1 - x_0)\big\rVert^2$ |

(The SD3 paper writes the same path with the time axis reversed — $z_t = (1-t)\,x_{\text{data}} + t\,\varepsilon$ — so its velocity has the opposite sign. Only the convention above is graded here.)

You will implement the interpolation, the training loss and an explicit **Euler** sampler, then train a small MLP velocity field on a 2-D mixture of 8 Gaussians and watch how the number of sampling steps matters.

---

### Requirements

1. **`interpolate(x0, x1, t)`** — `x0`, `x1` have the same shape `(N, ...)` (e.g. `(N, D)` or `(N, C, H, W)`); `t` has shape `(N,)`, one time per sample. Return $x_t = (1 - t)\,x_0 + t\,x_1$ with `t` broadcast over all trailing dims of its own row. Same shape as `x0`.
2. **`flow_matching_loss(model, x1, x0=None, t=None)`** — returns a scalar tensor (differentiable w.r.t. the model):
   - if `x0` is `None`, draw `x0 = torch.randn_like(x1)`; if `t` is `None`, draw `t = torch.rand(N)` — **independently per sample**, $t \sim U[0, 1)$;
   - build `x_t = interpolate(x0, x1, t)`, call `model(x_t, t)` **once** with `t` of shape `(N,)`;
   - return the **mean over all elements** of $(v - (x_1 - x_0))^2$ (i.e. `F.mse_loss`).
3. **`euler_sample(model, x0, num_steps)`** — integrate $\frac{dx}{dt} = $ `model(x, t)` from $t = 0$ to $t = 1$ with explicit Euler and uniform steps $\Delta t = 1/n$ (`n = num_steps`): for $k = 0, \dots, n-1$, evaluate the model at the **left endpoint** $t_k = k/n$ (passed as a `(N,)` tensor filled with $t_k$) and update $x \leftarrow x + \Delta t\, v$. Exactly `num_steps` model calls; never at $t = 1$. Return the final $x$; do **not** modify `x0` in place.

---

### Constraints

- ✅ Any PyTorch ops; `F.mse_loss` is fine.
- ✅ `model(x, t)` always receives `t` as a 1-D tensor with one entry per row of `x` — the same contract during training and sampling.
- ❌ No `torchdiffeq`, `diffusers` or other ODE / diffusion libraries.
- ❌ Don't hard-code 2-D inputs: `interpolate` must also work for image batches `(N, C, H, W)`.

---

<details>
  <summary>💡 Hint</summary>

  - For a fixed pair $(x_0, x_1)$ the path is a straight line traversed at constant speed, so its velocity does not depend on $t$.
  - A `(N,)` tensor times an `(N, D)` tensor broadcasts along the **last** dimension — which is wrong here and, when `N == D`, silently wrong. Reshape `t` to `(N, 1, ..., 1)`.
  - Euler with step size $1/n$ on $\dot x = -x$ gives $x_0(1 - 1/n)^n$, not $x_0 e^{-1}$: a nice way to test your sampler.

</details>

---

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

In [ ]:
def sample_8gaussians(n, radius=3.0, std=0.2, generator=None):
    """A 2-D mixture of 8 Gaussians on a circle. Returns (samples (n, 2), centers (8, 2))."""
    angles = torch.arange(8) * (2 * math.pi / 8)
    centers = radius * torch.stack([angles.cos(), angles.sin()], dim=1)
    idx = torch.randint(0, 8, (n,), generator=generator)
    return centers[idx] + std * torch.randn(n, 2, generator=generator), centers


class VelocityMLP(nn.Module):
    """v_theta(x, t) for 2-D data: an MLP on [x, t]. Given — the problem is the objective and the sampler."""

    def __init__(self, dim=2, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim + 1, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, dim),
        )

    def forward(self, x, t):                     # x: (N, dim), t: (N,)
        return self.net(torch.cat([x, t[:, None]], dim=-1))


data, centers = sample_8gaussians(4096, generator=torch.Generator().manual_seed(0))
print(f"data {tuple(data.shape)}, mean {data.mean(0).tolist()}, std {data.std(0).tolist()}")

In [ ]:
def interpolate(x0, x1, t):
    t = t.reshape(-1, *([1] * (x0.dim() - 1)))      # (N,) -> (N, 1, ..., 1): one t per row
    return (1 - t) * x0 + t * x1


def flow_matching_loss(model, x1, x0=None, t=None):
    if x0 is None:
        x0 = torch.randn_like(x1)                    # noise end of the path
    if t is None:
        t = torch.rand(x1.shape[0], device=x1.device, dtype=x1.dtype)   # t ~ U[0, 1) per sample
    x_t = interpolate(x0, x1, t)
    v = model(x_t, t)
    return F.mse_loss(v, x1 - x0)                    # target: d x_t / dt = x1 - x0


@torch.no_grad()
def euler_sample(model, x0, num_steps):
    x = x0.clone()                                   # never touch the caller's tensor
    dt = 1.0 / num_steps
    for k in range(num_steps):
        t = torch.full((x.shape[0],), k * dt, device=x.device, dtype=x.dtype)   # left endpoint
        x = x + dt * model(x, t)
    return x

In [ ]:
# Test 1: interpolate - endpoints and per-sample broadcasting (also for image batches)
x0, x1 = torch.randn(4, 4), torch.randn(4, 4)          # N == D on purpose
t = torch.tensor([0.0, 0.3, 0.7, 1.0])
xt = interpolate(x0, x1, t)
assert torch.allclose(xt[0], x0[0]) and torch.allclose(xt[3], x1[3]), "t=0 -> x0 (noise), t=1 -> x1 (data)"
assert torch.allclose(xt, (1 - t)[:, None] * x0 + t[:, None] * x1), "row i must use t[i]"
imgs0, imgs1 = torch.randn(4, 3, 5, 5), torch.randn(4, 3, 5, 5)
assert torch.allclose(interpolate(imgs0, imgs1, t), (1 - t).view(4, 1, 1, 1) * imgs0 + t.view(4, 1, 1, 1) * imgs1)
print("Test 1 passed: interpolate")

In [ ]:
# Test 2: the loss - reference value with explicit x0, t, and zero loss for the oracle velocity
torch.manual_seed(0)
x1 = torch.randn(32, 2) * 2 + 1
x0, t = torch.randn(32, 2), torch.rand(32)
toy = lambda x, tt: torch.tanh(x) * (1 + tt[:, None])  # any model(x, t)
ref = ((toy(interpolate(x0, x1, t), t) - (x1 - x0)) ** 2).mean()
assert torch.allclose(flow_matching_loss(toy, x1, x0=x0, t=t), ref), "loss != mean((v - (x1 - x0))^2)"
oracle = lambda x, tt: x1 - x0                           # knows the true pairing
assert flow_matching_loss(oracle, x1, x0=x0, t=t).item() < 1e-12, "the exact velocity must give zero loss"
# sampled x0, t: the conditional velocity (x1 - x_t) / (1 - t) is still an oracle
read_back = lambda x, tt: (x1 - x) / (1 - tt[:, None])
assert flow_matching_loss(read_back, x1).item() < 1e-6
print("Test 2 passed: flow_matching_loss")

In [ ]:
# Test 3: the Euler sampler
x0 = torch.randn(10, 2)
keep = x0.clone()
c = torch.tensor([0.5, -2.0])
assert torch.allclose(euler_sample(lambda x, t: c.expand_as(x), x0, 7), x0 + c, atol=1e-6), \
    "a constant velocity c must move x0 to x0 + c"
assert torch.equal(x0, keep), "euler_sample must not modify x0 in place"
for n in (1, 2, 5, 50):                                  # dx/dt = -x: Euler gives (1 - 1/n)^n
    out = euler_sample(lambda x, t: -x, x0, n)
    assert torch.allclose(out, x0 * (1 - 1 / n) ** n, atol=1e-6), f"n={n}"
seen = []
euler_sample(lambda x, t: (seen.append(t[0].item()), torch.zeros_like(x))[1], x0, 4)
assert len(seen) == 4 and all(abs(a - b) < 1e-6 for a, b in zip(seen, [0.0, 0.25, 0.5, 0.75])), \
    f"model must be called exactly at t_k = k / n = [0, 0.25, 0.5, 0.75], got {seen}"
print("Test 3 passed: euler_sample (exact for constant v, (1 - 1/n)^n for v = -x, left-endpoint grid)")

In [ ]:
# Train the velocity field on the 8-Gaussian mixture (~3-5 s on CPU)
torch.manual_seed(0)
model = VelocityMLP()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
for step in range(2000):
    x1, _ = sample_8gaussians(512)
    loss = flow_matching_loss(model, x1)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 500 == 0 or step == 1999:
        print(f"step {step:4d}  loss {loss.item():.3f}")
# The loss does NOT go to zero: many (x0, x1) pairs cross the same x_t with different
# velocities, and the best v is their conditional mean E[x1 - x0 | x_t]. Its floor is that variance.

In [ ]:
# Sample with different step counts: how many samples land near a mode?
torch.manual_seed(1)
z = torch.randn(2000, 2)
results = {}
for n in (1, 2, 4, 8, 32):
    xs = euler_sample(model, z, n)
    dist = torch.cdist(xs, centers).min(dim=1).values
    results[n] = (dist < 0.6).float().mean().item()          # within 3 std of a mode
    print(f"{n:3d} Euler steps: {results[n]:6.1%} of samples within 3 std of a mode, "
          f"sample std {xs.std(0).mean():.2f}")
assert results[32] > 0.85, "with 32 steps most samples should sit on a mode"
# One step from t = 0 gives x0 + v(x0, 0) ~= E[x1]: everything collapses to the middle. The marginal
# paths are straight per pair but curved on average, so a few more steps are needed - reflow and
# distillation straighten them further.

In [ ]:
# Optional plot (needs matplotlib)
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
if plt is not None:
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].scatter(*data[:2000].T, s=2)
    axes[0].set_title("data")
    for ax, n in zip(axes[1:], (1, 4, 32)):
        xs = euler_sample(model, z, n)
        ax.scatter(*xs.T, s=2)
        ax.set_title(f"{n} Euler steps")
    for ax in axes:
        ax.set_xlim(-4.5, 4.5)
        ax.set_ylim(-4.5, 4.5)
        ax.set_aspect("equal")
    plt.show()

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("flow-matching", interpolate=interpolate, flow_matching_loss=flow_matching_loss,
      euler_sample=euler_sample)

## Why it works

For one pair $(x_0, x_1)$ the path $x_t = (1-t)x_0 + t x_1$ has velocity $x_1 - x_0$. The model never sees the pair, only $(x_t, t)$, so the regression minimizer is the **conditional expectation**

$$v^*(x, t) = \mathbb{E}\,[\,x_1 - x_0 \mid x_t = x\,],$$

and Lipman et al. show this *marginal* velocity field generates exactly the marginal path $p_t$ = law of $x_t$ (continuity equation), which starts at $\mathcal N(0, I)$ and ends at $p_{\text{data}}$. So the conditional loss has the same gradients as the intractable "match the marginal velocity" loss, up to a constant — which is why the loss plateaus at a positive value (the conditional variance) even for a perfect model.

Sampling solves the ODE $\dot x = v_\theta(x, t)$ from $t = 0$; explicit Euler with step $\Delta t = 1/n$ evaluates $v$ at the start of each step, which is exact for constant velocity and first-order accurate otherwise (global error $O(1/n)$, e.g. $(1 - 1/n)^n$ vs $e^{-1}$ for $\dot x = -x$).

## Complexity

Training: one network evaluation per sample per step, like denoising score matching. Sampling: `num_steps` network evaluations (NFE) per batch, $O(n \cdot \text{cost}(v_\theta))$ time and $O(N \cdot \dim)$ memory (no trajectory stored).

## Common bugs

- **Broadcasting `t` along the last dim:** `(1 - t) * x0` with `t` of shape `(N,)` and `x0` of shape `(N, D)` crashes when `N != D` and is silently wrong when `N == D`. Reshape to `(N, 1, ...)`.
- **Sign / convention mix-ups:** target `x0 - x1`, or the SD3 time direction in the path but the Lipman direction in the sampler. Pick one convention and check `interpolate(x0, x1, 0) == x0`.
- **Uniform noise:** `torch.rand_like` instead of `torch.randn_like` for $x_0$ — the sampler then starts from a distribution the model never saw.
- **One `t` for the whole batch:** `torch.rand(1)` gives a high-variance gradient (every step trains a single time); draw one per sample.
- **Off-by-one time grid:** `torch.linspace(0, 1, n)` contains $t = 1$ and spacing $1/(n-1)$; `range(n + 1)` takes an extra step. Use $t_k = k/n$, $k < n$.
- **In-place update of `x0`** (`x = x0; x += dt * v`) corrupts the caller's noise — painful when comparing step counts from the same seed.

## Interview follow-ups

1. **How does this relate to diffusion?** With a Gaussian path, the flow-matching velocity is an affine function of the denoiser: for $x_t = \alpha_t x_1 + \sigma_t x_0$, $v = \dot\alpha_t\,\hat x_1 + \dot\sigma_t\,\hat\varepsilon$. Diffusion's probability-flow ODE is a flow with a particular (curved) schedule, and "v-prediction" (Salimans & Ho) is the velocity of the variance-preserving path $x = \cos\phi\, x_{\text{data}} + \sin\phi\,\varepsilon$ w.r.t. its angle $\phi$. Flow matching is mostly a cleaner parameterization + schedule, not a different model class.
2. **Why do straighter paths need fewer steps?** Euler is exact when the trajectory is a straight line traversed at constant speed. Each conditional path is straight, but the marginal flow averages crossing paths and bends; **reflow** (Liu et al.) re-pairs noise with the model's own samples and retrains so trajectories become nearly straight, enabling 1–4 step sampling; consistency/distillation methods push this further.
3. **Why does SD3 sample $t$ from a logit-normal instead of uniform?** The regression is hardest (and most useful for sample quality) at intermediate times; near the endpoints the target is nearly predictable from $x_t$. Logit-normal (and mode-heavy) densities put more training samples in the middle, which Esser et al. found improves FID/CLIP. SD3 also shifts the timestep schedule for higher resolutions, where more noise is needed to destroy the same amount of signal.
4. **How do you do classifier-free guidance with a flow model?** Exactly like diffusion: train with the condition dropped some fraction of the time and at sampling use $v = v_\varnothing + w\,(v_c - v_\varnothing)$; since the velocity is affine in the noise/data predictions, this is equivalent to guiding $\hat\varepsilon$.
5. **Why Euler and not a higher-order solver?** Euler is the baseline and makes the "straightness" argument transparent; in practice people use Heun / midpoint (2 NFE per step) or adaptive RK45 for evaluation, and the gains are largest when trajectories are curved.
6. **Can you compute likelihoods?** Yes — it is a continuous normalizing flow: $\log p_1(x_1) = \log p_0(x_0) - \int_0^1 \nabla\!\cdot v_\theta(x_t, t)\,dt$, with the divergence estimated by Hutchinson's trace estimator. Diffusion models can do the same through their probability-flow ODE.